In [ ]:
import ROOT, math, numpy as np
import random

In [ ]:
File = ROOT.TFile("LEP1MC1994_recons_aftercut-001.root")
File.ls()

In [ ]:
Canvas = ROOT.TCanvas("Canvas", "", 1024, 768)
Tree.Draw("nref")
Canvas.Draw()

In [ ]:
def GetPhi(x, y):
    if x == 0 and y > 0:
        return math.pi / 2

    if x == 0 and y < 0: 
        return (-(math.pi) / 2)

    if x > 0: 
        return np.arctan(y / x)

    if x < 0: 
        return math.pi + np.arctan(y / x)
    return 0

def jetshape(inputfilename, outputfilename, particletree1, particletree2):
    File = ROOT.TFile("LEP1MC1994_recons_aftercut-001.root")
    outFile = ROOT.TFile(outputfilename, "recreate")
    Hist12 = ROOT.TH1D("HIST12", "Total Energy vs. Distance", 80, 0, 20)
    Hist13 = ROOT.TH2D("HIST13", "E from tgen vs. E from t", 80, 0, 20, 80, 0, 20)
    Hist14 = ROOT.TH2D("HIST14", "polar theta from tgen vs. polar theta from t", 80, 0, 3.141, 80, -0.005, 0.005)
    Hist16 = ROOT.TH2D("HIST16", "azimuth from tgen vs. azimuth from t", 80, -3.141 / 2, 3.141 * 1.5, 80,-.05, 0.05)
    Counter = 0
    TreeParticle1 = File.Get(particletree1)
    TreeParticle2 = File.Get(particletree2)
    TotalNumberOfCollisions = TreeParticle1.GetEntries("nParticle")
    TargetNumberOfCollisions = TotalNumberOfCollisions * 0.01
    CurrentlyProcessed = 0
    for Event1, Event2 in zip(TreeParticle1, TreeParticle2):
         CurrentlyProcessed = CurrentlyProcessed + 1
         if CurrentlyProcessed > TargetNumberOfCollisions:
             break
        GenPX = [Event1.px[i] for i in range(Event1.nParticle)]
         GenPY = [Event1.py[i] for i in range(Event1.nParticle)]
         GenPZ = [Event1.pz[i] for i in range(Event1.nParticle)]
         GenP =  [math.sqrt(Event1.px[i] ** 2 + Event1.py[i] ** 2 + Event1.pz[i] ** 2) for i in range(Event1.nParticle)]
         GenE = [math.sqrt(Event1.mass[i]**2 + GenP[i]**2) for i in range(Event1.nParticle)]
         Gentheta = np.arccos([GenPZ[i] / math.sqrt(Event1.px[i] ** 2 + Event1.py[i] ** 2 + Event1.pz[i] ** 2) for i in range(Event1.nParticle)])
        #E1 Event1.charge[...] == 0 Event2.charge[...' ==0 all plots stay the same just adding a qualifier
         PGPX = [Event2.px[i] for i in range(Event2.nParticle)]
         PGPY = [Event2.py[i] for i in range(Event2.nParticle)]
         #print([ParticleGen.eta[i] for i in range(ParticleGen.nParticle)])
         PGPZ = [Event2.pz[i] for i in range(Event2.nParticle)]
         PGP = [math.sqrt(Event2.px[i] ** 2 + Event2.py[i] ** 2 + Event2.pz[i] ** 2) for i in range(Event2.nParticle)]
         PGE = [math.sqrt(Event2.mass[i]**2 + PGP[i]**2) for i in range(Event2.nParticle)]
         Ptheta = np.arccos([PGPZ[i] / math.sqrt(Event2.px[i] ** 2 + Event2.py[i] ** 2 + Event2.pz[i] ** 2) for i in range(Event2.nParticle)])
         #E1prime
         for iG in range(len(GenPX)):
             # we want to do both <40 and >40 want as well 10-20; 20-30; 30-40; 40+ (R8; MCt v. MCtgen) (MCtgen and Datat)
             if GenE [iG]< 1: 
                 continue
             Counter = Counter + 1
             Bestsofar = 0 
             Bestdistance = 9999
             for iP in range(Event2.nParticle):
                 if PGP[iP] == 0 or GenP[iG] == 0:
                     continue
                 Cosine = (((GenPX[iG] * PGPX[iP]) + (GenPY[iG] * PGPY[iP]) + (GenPZ[iG] * PGPZ[iP])) / (GenP[iG] * PGP[iP]))
                 if Cosine > 1: Cosine = 1
                 if Cosine < -1: Cosine = -1
                 Distance = np.arccos(Cosine)
                 if Distance < Bestdistance :
                     Bestsofar = iP
                     Bestdistance = Distance
                 Hist12.GetXaxis().SetRangeUser(0, 4)
             Hist12.Fill(Bestsofar)
             if Bestdistance < 0.04:
                 #Hist14.Fill(Gentheta[iG],Ptheta[Bestsofar] - Gentheta[iG])
                 #Hist16.Fill(GetPhi(GenPX[iG], GenPY[iG]),GetPhi( PGPX[Bestsofar], PGPY[Bestsofar]))
                 Hist16.Fill(GetPhi(GenPX[iG], GenPY[iG]),GetPhi( PGPX[Bestsofar], PGPY[Bestsofar])- GetPhi(GenPX[iG], GenPY[iG]))
    Hist12.Scale(1 / Counter)
    for i in range (0, 80):
        r = Hist12.GetXaxis(). GetBinLowEdge(i + 1)
        R = Hist12.GetXaxis().GetBinUpEdge(i + 1)
        x = Hist12.GetBinContent(i + 1)
        Area = math.pi*(R*R - r*r)
        avg = x / Area
        Hist12.SetBinContent(i + 1, avg)
    
    Canvas = ROOT.TCanvas("Canvas", "", 1024, 768)
    #Hist2.GetXaxis().SetRangeUser(0, 0.8)
    #Hist3.GetXaxis().SetRangeUser(0, 0.8)
    Hist12.Draw()
    #Canvas.SetLogy()
    #delete previous line when doing new type of running
    outFile.cd()
    Hist12.Write()
    Hist16.Write()
    outFile.Close()

In [ ]:
jetshape("LEP1MC1994_recons_aftercut-001.root", "MCbestdistancephi.root", "tgen", "t")

In [ ]:
Canvas = ROOT.TCanvas("Canvas", "", 1024, 768)
#Canvas.SetLogy()
File1 = ROOT.TFile("MCbestdistancephi.root")
#different simulation matchbox vs. sherpa vs. LEPIMC
#with and without detector effects which is LEPIMC with t and tgen two histograms
#everything under LEPIMC ALEPHMC
# with detector vs. data LEPIMC vs. LEPIData 
#File2 = ROOT.TFile("reconLEPMCtgenR8.root")
#File3 = ROOT.TFile("reconLEPMCtR8.root")

Hist1 = File1.Get("HIST16")
#Hist2 = File2.Get("HIST12")
#Hist3 = File3.Get("HIST12")

Hist1.SetLineColor(ROOT.kBlue)
#Hist2.SetLineColor(ROOT.kRed)
#Hist3.SetLineColor(ROOT.kGreen)

Legend = ROOT.TLegend(1, 1, 1, 0.6)
Legend.AddEntry(Hist1, "LEPMCtR8", "lp")
#Legend.AddEntry(Hist2, "LEPMCtgenR8", "lp")
#Legend.AddEntry(Hist3, "LEPMCR8", "lp")
Hist16.projectiony

Hist1.SetMarkerStyle(20)
Hist1.SetMarkerColor(ROOT.kBlue)
#Hist2.SetMarkerStyle(20)
#Hist2.SetMarkerColor(ROOT.kRed)
#Hist3.SetMarkerStyle(20)
#Hist3.SetMarkerColor(ROOT.kGreen)
Hist1.Draw("colz");
#Hist2.Draw("same")
#Hist3.Draw("same")

#Do everything with R8 and <0.8 distance 
#
Canvas.Draw()
Legend.Draw()